### Tasks:
1. Calculate **IQR bounds** for `transaction_amount` and identify outlier indices.
2. Calculate **Z-Score** values and flag extreme values ($|Z| > 3.0$).
3. Fit an **Isolation Forest** to identify multi-variable anomalies across `transaction_amount` and `risk_score`.
4. Apply **Winsorization (capping)** to restrict outliers within IQR boundaries.

In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

np.random.seed(101)

# Sample Data Generation
data_size = 500
tx_amounts = np.random.exponential(scale=50, size=data_size)
risk_scores = np.random.normal(loc=50, scale=10, size=data_size)

# Inject anomalies
tx_amounts[:15] = np.random.uniform(300, 800, size=15)
risk_scores[:15] = np.random.uniform(90, 100, size=15)

df = pd.DataFrame({
    'transaction_amount': tx_amounts,
    'risk_score': risk_scores
})
df.head()

,transaction_amount,risk_score
0,735.961428,98.005781
1,356.679957,94.201870
2,582.949562,90.045716
3,634.725394,90.151992
4,356.890655,91.551146


### Task 1: Complete the IQR Outlier Function
Fill in the missing mathematical logic for calculating $Q1$, $Q3$, $IQR$, and the lower/upper bounds.

In [3]:
def get_iqr_outliers(series: pd.Series, k: float = 1.5):
    # TODO: Calculate q1, q3, iqr, lower_bound, upper_bound
    q1 = df["transaction_amount"].quantile(0.25)
    q3 = df["transaction_amount"].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - k * iqr
    upper_bound = q3 + k * iqr
    
    outlier_mask = (series < lower_bound) | (series > upper_bound)
    return outlier_mask, lower_bound, upper_bound

# Test your function
mask, lb, ub = get_iqr_outliers(df['transaction_amount'])
print(f"Detected {mask.sum()} outliers using IQR.")

Detected 32 outliers using IQR.


In [4]:
# Z-scores
df["z-score"] = (df["transaction_amount"] - df["transaction_amount"].mean()) / df["transaction_amount"].std()

df["z_outlier"] = df["z-score"].abs() > 3.0

print("Extreme values detected using Z-Score:")
print(df[df["z_outlier"]])

print("\nTotal Z-Score outliers:", df["z_outlier"].sum())

Extreme values detected using Z-Score:
    transaction_amount  risk_score   z-score  z_outlier
0           735.961428   98.005781  6.681068       True
2           582.949562   90.045716  5.156194       True
3           634.725394   90.151992  5.672177       True
6           740.879356   91.015839  6.730079       True
7           626.556279   92.884447  5.590766       True
8           699.133795   99.329601  6.314054       True
9           387.450952   93.867839  3.207908       True
11          474.468873   94.201265  4.075105       True
12          443.879589   93.747241  3.770260       True
13          770.730753   98.721893  7.027570       True
14          584.870222   99.053423  5.175334       True

Total Z-Score outliers: 11


### Task 2: Isolation Forest Detection
Fit an `IsolationForest` estimator with `contamination=0.03` on both features (`transaction_amount` and `risk_score`).

In [5]:
# TODO: Instantiate and fit IsolationForest
# iso = IsolationForest(...)
# df['is_outlier_iforest'] = ...

X = df[["transaction_amount", "risk_score"]]

iso = IsolationForest(
    contamination=0.03,
    random_state=42
)

predictions = iso.fit_predict(X)

df["is_outlier_iforest"] = predictions == -1

print(df[["transaction_amount", "risk_score", "is_outlier_iforest"]])

print("\nOutliers detected:", (predictions == -1).sum())
print("Normal transactions:", (predictions == 1).sum())


     transaction_amount  risk_score  is_outlier_iforest
0            735.961428   98.005781                True
1            356.679957   94.201870                True
2            582.949562   90.045716                True
3            634.725394   90.151992                True
4            356.890655   91.551146                True
..                  ...         ...                 ...
495            0.864287   46.380685               False
496           71.955157   57.547495               False
497           17.084440   35.382111               False
498           41.324163   56.847332               False
499            4.335372   46.192489               False

[500 rows x 3 columns]

Outliers detected: 15
Normal transactions: 485
